In [1]:
import pandas as pd
import json
import numpy as np

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/brainlat"

In [3]:
test = pd.read_csv(f"/projectnb/vkolagrp/ketanss/adrdfm-2/brainlat_fromscratch/brainlat_mri_cohort_oct_16.csv")

In [4]:
test.head()

,MRI_ID,Diagnosis,Metadata,EEG Report,MRI Report,input_json
0,sub-MXA00014,Healthy,"{""Sex"": ""Female"", ""Age"": 66.0, ""Years of educa...",NaN,NaN,"{""Metadata"": {""Sex"": ""Female"", ""Age"": 66.0, ""Y..."
1,sub-MXA00017,Healthy,"{""Sex"": ""Female"", ""Age"": 64.0, ""Years of educa...",NaN,NaN,"{""Metadata"": {""Sex"": ""Female"", ""Age"": 64.0, ""Y..."
2,sub-CLA00026,Alzheimer's disease,"{""Sex"": ""Female"", ""Age"": 81.0, ""Years of educa...",NaN,"{""MRI features extraction procedure"": ""A Gener...","{""Metadata"": {""Sex"": ""Female"", ""Age"": 81.0, ""Y..."
3,sub-CLA00045,Behavioral variant frontotemporal dementia,"{""Sex"": ""Female"", ""Age"": 57.0, ""Years of educa...",NaN,"{""MRI features extraction procedure"": ""A Gener...","{""Metadata"": {""Sex"": ""Female"", ""Age"": 57.0, ""Y..."
4,sub-CLA00057,Alzheimer's disease,"{""Sex"": ""Female"", ""Age"": 60.0, ""Years of educa...",NaN,"{""MRI features extraction procedure"": ""A Gener...","{""Metadata"": {""Sex"": ""Female"", ""Age"": 60.0, ""Y..."


In [5]:
json.loads(test.iloc[2]['input_json'])

{'Metadata': {'Sex': 'Female',
  'Age': 81.0,
  'Years of education': 4.0,
  'Laterality': 'Right',
  'Total score of MoCa (0–30)': 15.0,
  'MoCa visuospatial assessment (0–5)': 3.0,
  'MoCa recognition assessment (0–3)': 3.0,
  'MoCa attention assessment (0–6)': 4.0,
  'MoCa language assessment (0–3)': 2.0,
  'MoCa abstraction assessment (0–2)': 2.0,
  'MoCa memory assessment (0–5)': 0.0,
  'MoCa orientation assessment (0–6)': 1.0},
 'MRI Report': {'MRI features extraction procedure': 'A Generalized Additive Model for Location, Scale, and Shape (GAMLSS) was fitted to FastSurfer-derived regional brain volumes acquired from 2,087 healthy controls, following the Brain charts for the human lifespan methodology published in Nature (2022). Data were aggregated from five independent cohorts, and the analysis included region-of-interest (ROI) volumes for the medial and lateral temporal lobes, medial and lateral parietal lobes, frontal, occipital lobes, as well as the inferior lateral ventricl

In [6]:
def modify_json(row):
    input_json = json.loads(row['input_json'])
    if 'MRI Report' in input_json:
        input_json['Imaging evidence'] = input_json['MRI Report']
        input_json.pop('MRI Report')
    
    row['input_json'] = json.dumps(input_json)
    return row


test = test.apply(modify_json, axis=1)
test['patient_summary'] = test['input_json']
test['ID'] = test['MRI_ID']

test.drop(['MRI_ID', 'input_json'], axis=1, inplace=True)


In [7]:
test.head()

,Diagnosis,Metadata,EEG Report,MRI Report,patient_summary,ID
0,Healthy,"{""Sex"": ""Female"", ""Age"": 66.0, ""Years of educa...",NaN,NaN,"{""Metadata"": {""Sex"": ""Female"", ""Age"": 66.0, ""Y...",sub-MXA00014
1,Healthy,"{""Sex"": ""Female"", ""Age"": 64.0, ""Years of educa...",NaN,NaN,"{""Metadata"": {""Sex"": ""Female"", ""Age"": 64.0, ""Y...",sub-MXA00017
2,Alzheimer's disease,"{""Sex"": ""Female"", ""Age"": 81.0, ""Years of educa...",NaN,"{""MRI features extraction procedure"": ""A Gener...","{""Metadata"": {""Sex"": ""Female"", ""Age"": 81.0, ""Y...",sub-CLA00026
3,Behavioral variant frontotemporal dementia,"{""Sex"": ""Female"", ""Age"": 57.0, ""Years of educa...",NaN,"{""MRI features extraction procedure"": ""A Gener...","{""Metadata"": {""Sex"": ""Female"", ""Age"": 57.0, ""Y...",sub-CLA00045
4,Alzheimer's disease,"{""Sex"": ""Female"", ""Age"": 60.0, ""Years of educa...",NaN,"{""MRI features extraction procedure"": ""A Gener...","{""Metadata"": {""Sex"": ""Female"", ""Age"": 60.0, ""Y...",sub-CLA00057


In [8]:
json.loads(test.iloc[2]['patient_summary'])

{'Metadata': {'Sex': 'Female',
  'Age': 81.0,
  'Years of education': 4.0,
  'Laterality': 'Right',
  'Total score of MoCa (0–30)': 15.0,
  'MoCa visuospatial assessment (0–5)': 3.0,
  'MoCa recognition assessment (0–3)': 3.0,
  'MoCa attention assessment (0–6)': 4.0,
  'MoCa language assessment (0–3)': 2.0,
  'MoCa abstraction assessment (0–2)': 2.0,
  'MoCa memory assessment (0–5)': 0.0,
  'MoCa orientation assessment (0–6)': 1.0},
 'Imaging evidence': {'MRI features extraction procedure': 'A Generalized Additive Model for Location, Scale, and Shape (GAMLSS) was fitted to FastSurfer-derived regional brain volumes acquired from 2,087 healthy controls, following the Brain charts for the human lifespan methodology published in Nature (2022). Data were aggregated from five independent cohorts, and the analysis included region-of-interest (ROI) volumes for the medial and lateral temporal lobes, medial and lateral parietal lobes, frontal, occipital lobes, as well as the inferior lateral ve

In [9]:
test['Diagnosis'].value_counts()

Diagnosis
Alzheimer's disease                           269
Healthy                                       242
Behavioral variant frontotemporal dementia    149
Parkinson's disease                            53
Multiple sclerosis                             26
Name: count, dtype: int64

In [10]:
test = test[(test['Diagnosis'] != "Multiple sclerosis") & (test['Diagnosis'] != "Parkinson's disease")].sample(frac=1, random_state=42).reset_index(drop=True)

In [11]:
test['Diagnosis'].value_counts()

Diagnosis
Alzheimer's disease                           269
Healthy                                       242
Behavioral variant frontotemporal dementia    149
Name: count, dtype: int64

In [12]:
test.to_csv(f'{directory_path}/test_gamlss.csv', index=False)

In [13]:
len(test)

660